ETAPE 1: nettoyage et visualisation du jeu de données
ce premier notebook documente la démarche d'exploration et de nettoyage du dataset fourni volontairement eronné

In [4]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("risk_monitor_dataset.sqlite")
#voir les tables du dataset
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

            name
0          users
1  subscriptions
2    memberships
3       payments
4     complaints


ce code nous permet de charger les 3 premières lignes de chaque table afin d'avoir un apperçu du dataset et des anomalies


In [5]:
df_users = pd.read_sql("SELECT * FROM users", conn)
df_subscriptions = pd.read_sql("SELECT * FROM subscriptions", conn)
df_memberships = pd.read_sql("SELECT * FROM memberships", conn)
df_payments = pd.read_sql("SELECT * FROM payments", conn)
df_complaints = pd.read_sql("SELECT * FROM complaints", conn)

# Résumé rapide
for name, df in [("users", df_users), ("subscriptions", df_subscriptions),
                  ("memberships", df_memberships), ("payments", df_payments),
                  ("complaints", df_complaints)]:
    print(f"\n{'='*50}")
    print(f"TABLE : {name.upper()}")
    print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print(f"Colonnes : {df.columns.tolist()}")
    print(df.head(3))


TABLE : USERS
Dimensions : 2001 lignes × 8 colonnes
Colonnes : ['id', 'email', 'country', 'signup_date', 'status', 'last_seen', 'referral_code', 'phone_prefix']
   id               email country           signup_date  status  \
0   1     user_1@yahoo.fr      IT   2021-02-09 21:26:47     1.0   
1   2    user_2@gmail.com      CH  2022-01-09T06:49:44Z     0.0   
2   3  user_3@outlook.com      BE            1584765297     1.0   

             last_seen referral_code phone_prefix  
0  2021-04-28 14:23:00          None          +49  
1  2023-01-05 17:03:20          None          +41  
2  2021-07-30 05:38:37      27cb14dc          +32  

TABLE : SUBSCRIPTIONS
Dimensions : 400 lignes × 8 colonnes
Colonnes : ['id', 'brand', 'owner_id', 'created_at', 'status', 'max_slots', 'price_cents', 'currency']
   id          brand  owner_id           created_at  status  max_slots  \
0   1  Microsoft 365         1  2023-01-14 23:23:57       0          4   
1   2        HBO Max      1670  2023-07-16 21:39:5

A l'aide du logiciel DB browser for SQLite, j'ai pu visualiser l'entièreté du dataset et remarquer certaines anomalies :
- syntaxe différente : majuscule/minuscule;
- langue différente : anglais/francais;
- contenu des cellules : "NULL"/valeur manquantes/ "refunded";
- les formats d'écriture des dates & heure;
- le préfixe téléphonique et le pays ne concordent pas;
- supposément des doublons;


correction souhaitée des anomalies détéctées afin que ca ne fausse pas la suite et que python ne considère pas "EUR" et "eur" différement par exemple :
- cellule incohérente (EUR/eur) -> normalisation en minuscules  
- mélange français/anglais ->| mapping vers valeurs anglaises standard
- préfixe téléphonique =! pays -> flag de l'incohérence, pas suppression
- 3 formats de dates différents -> normalisation en datetime UTC
- NULL textuels ("None", "NULL", "") -> Remplacement par `np.nan`
- doublons partiels -> détection et suppression des doublons exacts


On traite les anomalies dans un ordre logique :
1. d'abord les faux NULL car ils faussent toutes les analyses suivantes
2. ensuite la casse pour éviter les doublons logiques (EUR =! eur pour Python)
3. puis la langue pour homogénéiser les catégories
4. Les dates — pour rendre les calculs temporels possibles
5. Les incohérences pays/téléphone — flaggées car signal de fraude potentiel
6. Les doublons exacts — supprimés en dernier pour ne pas perdre de données utiles

chaque colonne nettoyée crée une nouvelle colonne `_clean`
qui conserve la valeur originale pour traçabilité, on ne supprime pas les data d'origines !

In [15]:
import sqlite3
import pandas as pd
import numpy as np

print("🧹 DÉBUT DU NETTOYAGE\n")

print("ÉTAPE 1 — Remplacement des faux NULL...")

FAUX_NULLS = ["None", "NULL", "null", "none", "N/A", "n/a", ""]

for df in [df_users, df_subscriptions, df_memberships, df_payments, df_complaints]:
    df.replace(FAUX_NULLS, np.nan, inplace=True)

print("✅ Faux NULL remplacés par NaN")

print("\nÉTAPE 2 — Normalisation de la casse...")

# Currencies
df_payments['currency'] = df_payments['currency'].str.upper().str.strip()
df_subscriptions['currency'] = df_subscriptions['currency'].str.upper().str.strip()

# Pays
df_users['country'] = df_users['country'].str.upper().str.strip()

# Status complaints
df_complaints['status'] = df_complaints['status'].str.lower().str.strip()

# Type complaints
df_complaints['type'] = df_complaints['type'].str.lower().str.strip()

print("✅ Casse normalisée")

print("\nÉTAPE 3 — Normalisation des langues...")

mapping_type = {
    "accès refusé"        : "access_denied",
    "acces refusé"        : "access_denied",
    "acces refuse"        : "access_denied",
    "paiement échoué"     : "payment_failed",
    "paiement echoue"     : "payment_failed",
    "remboursement"       : "refund_request",
    "fraude"              : "fraud",
    "abonnement inactif"  : "subscription_inactive",
}

df_complaints['type_clean'] = df_complaints['type'].replace(mapping_type)

print("Mapping appliqué :")
print(df_complaints[['type', 'type_clean']].drop_duplicates().to_string())

print("\nÉTAPE 4 — Normalisation des dates...")

def parse_date_any_format(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    try:
        # Unix timestamp (ex: 1584765297)
        if val.isdigit() and len(val) == 10:
            return pd.Timestamp(int(val), unit='s')
        # ISO 8601 avec timezone (ex: 2022-01-09T06:49:44Z)
        elif "T" in val:
            return pd.to_datetime(val, utc=True).tz_localize(None)
        # Format standard
        else:
            return pd.to_datetime(val)
    except:
        return pd.NaT

df_users['signup_date_clean'] = df_users['signup_date'].apply(parse_date_any_format)
df_users['last_seen_clean'] = df_users['last_seen'].apply(parse_date_any_format)
df_payments['created_at_clean'] = df_payments['created_at'].apply(parse_date_any_format)
df_payments['captured_at_clean'] = df_payments['captured_at'].apply(parse_date_any_format)
df_memberships['joined_at_clean'] = df_memberships['joined_at'].apply(parse_date_any_format)
df_memberships['left_at_clean'] = df_memberships['left_at'].apply(parse_date_any_format)

invalides = df_users['signup_date_clean'].isna().sum()
print(f"✅ Dates normalisées | Dates invalides users : {invalides}")

print("\nÉTAPE 5 — Détection incohérences pays/téléphone...")

COUNTRY_PREFIX = {
    'FR': '+33', 'BE': '+32', 'CH': '+41', 'DE': '+49',
    'IT': '+39', 'ES': '+34', 'GB': '+44', 'NL': '+31',
    'PT': '+351', 'LU': '+352', 'AT': '+43', 'US': '+1',
}

def check_prefix(row):
    country = row['country']
    prefix = row['phone_prefix']
    if pd.isna(country) or pd.isna(prefix):
        return 'unknown'
    expected = COUNTRY_PREFIX.get(country)
    if expected is None:
        return 'country_not_mapped'
    return 'ok' if prefix == expected else 'mismatch'

df_users['prefix_flag'] = df_users.apply(check_prefix, axis=1)
print(df_users['prefix_flag'].value_counts())

print("\nÉTAPE 6 — Suppression des doublons...")

for name, df in [("users", df_users), ("subscriptions", df_subscriptions),
                  ("memberships", df_memberships), ("payments", df_payments),
                  ("complaints", df_complaints)]:
    avant = len(df)
    df.drop_duplicates(inplace=True)
    apres = len(df)
    print(f"  {name} : {avant - apres} doublons supprimés ({avant} → {apres} lignes)")

print("\n✅ NETTOYAGE TERMINÉ")
print(f"\nTables nettoyées :")
for name, df in [("users", df_users), ("subscriptions", df_subscriptions),
                  ("memberships", df_memberships), ("payments", df_payments),
                  ("complaints", df_complaints)]:
    print(f"  {name} : {len(df)} lignes")

🧹 DÉBUT DU NETTOYAGE

ÉTAPE 1 — Remplacement des faux NULL...
✅ Faux NULL remplacés par NaN

ÉTAPE 2 — Normalisation de la casse...
✅ Casse normalisée

ÉTAPE 3 — Normalisation des langues...
Mapping appliqué :
                       type             type_clean
1011          access_denied          access_denied
656      owner_unresponsive     owner_unresponsive
886         fraud_suspicion        fraud_suspicion
127       wrong_credentials      wrong_credentials
1044                  other                  other
552            accès refusé          access_denied
304           billing_issue          billing_issue
185   subscription_inactive  subscription_inactive

ÉTAPE 4 — Normalisation des dates...


/tmp/ipykernel_8756/3585830562.py:78: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val)


✅ Dates normalisées | Dates invalides users : 0

ÉTAPE 5 — Détection incohérences pays/téléphone...
prefix_flag
ok                    1148
unknown                475
mismatch               246
country_not_mapped     132
Name: count, dtype: int64

ÉTAPE 6 — Suppression des doublons...
  users : 0 doublons supprimés (2001 → 2001 lignes)
  subscriptions : 0 doublons supprimés (400 → 400 lignes)
  memberships : 0 doublons supprimés (1083 → 1083 lignes)
  payments : 0 doublons supprimés (7277 → 7277 lignes)
  complaints : 0 doublons supprimés (1211 → 1211 lignes)

✅ NETTOYAGE TERMINÉ

Tables nettoyées :
  users : 2001 lignes
  subscriptions : 400 lignes
  memberships : 1083 lignes
  payments : 7277 lignes
  complaints : 1211 lignes


246 mismatches pays/préfixe — des gens avec un numéro allemand mais déclarés en France par exemple. potentiellement des fraudes ..

le sujet mentionait d'éventuels doublonsAprès le premier nettoyage, on inspecte toutes les valeurs uniques de chaque colonne sensible.
comme le premier nettoyage était basé sur un aperçu de 3 lignes par table il était insuffisant pour couvrir toutes les variantes.

Cette vérification révèle les anomalies manquées :
- 115 valeurs encore en français dans `complaints.type`
- 3 variantes de "succeeded" dans `payments.status` (`success`, `suceeded`, `Succeeded`)
- Des `"NaN"` textuels résiduels dans `memberships.reason`
- Des codes numériques non documentés dans `users.status`


In [7]:
print("RECHERCHE DOUBLONS PARTIELS\n")

# Même user dans la même subscription plusieurs fois
dupes_memberships = df_memberships[
    df_memberships.duplicated(subset=['user_id', 'subscription_id'], keep=False)
]
print(f"Memberships — même user + même subscription : {len(dupes_memberships)} lignes")
if len(dupes_memberships) > 0:
    print(dupes_memberships.sort_values(['user_id','subscription_id']).head(10))

# Même paiement en double (même user, même montant, même date)
dupes_payments = df_payments[
    df_payments.duplicated(subset=['user_id', 'subscription_id', 'amount_cents', 'created_at'], keep=False)
]
print(f"\nPayments — même user + montant + date : {len(dupes_payments)} lignes")
if len(dupes_payments) > 0:
    print(dupes_payments.head(10))

# Même réclamation en double
dupes_complaints = df_complaints[
    df_complaints.duplicated(subset=['reporter_id', 'target_id', 'type_clean'], keep=False)
]
print(f"\nComplaints — même reporter + target + type : {len(dupes_complaints)} lignes")
if len(dupes_complaints) > 0:
    print(dupes_complaints.head(10))

RECHERCHE DOUBLONS PARTIELS

Memberships — même user + même subscription : 0 lignes

Payments — même user + montant + date : 51 lignes
      id  user_id  subscription_id  amount_cents  fee_cents     status  \
21    22      590              168           727         88    pending   
33    34     1282              335           642         66     failed   
52    53     1990              244           303         39  succeeded   
91    92       79              379           828         66    pending   
92    93       79              379           224         25  succeeded   
98    99      720              398           657         51     failed   
101  102      720              398           366         42  succeeded   
117  118     1203               48           517         36  Succeeded   
198  199     1073              394           588         73     failed   
206  207     1446              172           469         51  succeeded   

                    created_at          captured_a

Avec ce code on voit qu'il y a encore des anomalies ex. "Succeeded" ou le mot n'est pas intégralement en majuscule ou minuscule .. il faudra aussi corriger ca !
On rq également dans la table "complaints" qu'un user s'auto plain de lui

## 7. Traitement des doublons partiels

Les doublons "exacts" (ligne identique) ont déjà été supprimés.
On cherche maintenant les doublons partiels : même entité,
informations contradictoires ou répétées avec des variantes.

Deux cas détectés :
- Payments : même user + même subscription + même montant mais dates
  ou statuts différents -> ce sont des tentatives de paiement retentées, on les garde mais on les flaggue `is_retry = True`
  car ça peut servir lors du scoring.
- Complaints : même reporter, même target, même type → doublon réel.
  On garde la plus récente et on supprime l'ancienne. Cas particulier :
  32 users qui se plaignent d'eux-mêmes (`reporter_id == target_id`) —
  flaggués `self_complaint = True` car signal de comportement suspect.

In [8]:
#code pour nettoyer les doublons partiels
print("🧹 TRAITEMENT DES DOUBLONS PARTIELS\n")

# ── PAYMENTS ──────────────────────────────────────────────
# Normaliser la casse du status (Succeeded → succeeded)
df_payments['status'] = df_payments['status'].str.lower().str.strip()

# Flagger les paiements en doublon partiel (même user/sub/montant, dates différentes)
df_payments['is_retry'] = df_payments.duplicated(
    subset=['user_id', 'subscription_id', 'amount_cents'], keep=False
)
nb_retries = df_payments['is_retry'].sum()
print(f"Payments flaggés comme 'retry' : {nb_retries} lignes")

# ── COMPLAINTS ────────────────────────────────────────────
# Cas 1 : reporter_id == target_id (user qui se plaint de lui-même)
df_complaints['self_complaint'] = df_complaints['reporter_id'] == df_complaints['target_id']
nb_self = df_complaints['self_complaint'].sum()
print(f"\nComplaints — user se plaint de lui-même : {nb_self} cas")
print(df_complaints[df_complaints['self_complaint']][['id','reporter_id','target_id','type_clean','status']])

# Cas 2 : vrais doublons partiels (même reporter/target/type, dates différentes)
# On garde la plus récente, on supprime l'ancienne
df_complaints = df_complaints.sort_values('created_at', ascending=False)
df_complaints['is_dupe_complaint'] = df_complaints.duplicated(
    subset=['reporter_id', 'target_id', 'type_clean'], keep='first'
)
nb_dupes = df_complaints['is_dupe_complaint'].sum()
print(f"\nComplaints — doublons partiels supprimés : {nb_dupes}")
df_complaints = df_complaints[~df_complaints['is_dupe_complaint']].copy()
print(f"Complaints restantes : {len(df_complaints)}")

print("\n✅ Doublons partiels traités")

🧹 TRAITEMENT DES DOUBLONS PARTIELS

Payments flaggés comme 'retry' : 89 lignes

Complaints — user se plaint de lui-même : 32 cas
        id  reporter_id  target_id             type_clean       status
16      17          636        636        fraud_suspicion  in_progress
71      72         1204       1204          access_denied    escalated
75      76          439        439          access_denied    escalated
94      95          500        500  subscription_inactive       closed
133    134          702        702        fraud_suspicion     resolved
165    166          439        439          access_denied       closed
180    181         1112       1112          billing_issue     resolved
236    237         1579       1579     owner_unresponsive  in_progress
271    272         1948       1948          billing_issue         open
294    295          924        924          billing_issue     resolved
327    328         1072       1072        fraud_suspicion         open
332    333         

In [9]:
#code pour enregistrer les nouvelles table du dataset, nettoyées, sur un fichier crée en local
import os

os.makedirs('../data', exist_ok=True)

df_users.to_csv('../data/users_clean.csv', index=False)
df_subscriptions.to_csv('../data/subscriptions_clean.csv', index=False)
df_memberships.to_csv('../data/memberships_clean.csv', index=False)
df_payments.to_csv('../data/payments_clean.csv', index=False)
df_complaints.to_csv('../data/complaints_clean.csv', index=False)

print("✅ Données nettoyées exportées dans /data/")
print("\nRécapitulatif final :")
for name, df in [("users", df_users), ("subscriptions", df_subscriptions),
                  ("memberships", df_memberships), ("payments", df_payments),
                  ("complaints", df_complaints)]:
    print(f"  {name}_clean.csv : {len(df)} lignes, {len(df.columns)} colonnes")

✅ Données nettoyées exportées dans /data/

Récapitulatif final :
  users_clean.csv : 2001 lignes, 11 colonnes
  subscriptions_clean.csv : 400 lignes, 8 colonnes
  memberships_clean.csv : 1083 lignes, 9 colonnes
  payments_clean.csv : 7277 lignes, 13 colonnes
  complaints_clean.csv : 1211 lignes, 12 colonnes


en regardant les nouveaux fichiés .csv (donc les tables nettoyées), on rq encore des anomalies .., le code ci dessous nous aide à voir les anomalies encore présentes pour nettoyer ça

In [10]:
#code afin de voir ce qu'on a loupé dans les anomalies pour re corriger ça
print("🔍 TOUTES LES VALEURS UNIQUES PAR COLONNE\n")

print("=== COMPLAINTS — type ===")
print(df_complaints['type'].value_counts(dropna=False).to_string())

print("\n=== COMPLAINTS — status ===")
print(df_complaints['status'].value_counts(dropna=False).to_string())

print("\n=== MEMBERSHIPS — reason ===")
print(df_memberships['reason'].value_counts(dropna=False).to_string())

print("\n=== PAYMENTS — status ===")
print(df_payments['status'].value_counts(dropna=False).to_string())

print("\n=== USERS — status ===")
print(df_users['status'].value_counts(dropna=False).to_string())

🔍 TOUTES LES VALEURS UNIQUES PAR COLONNE

=== COMPLAINTS — type ===
type
access_denied            268
other                    153
wrong_credentials        145
subscription_inactive    143
owner_unresponsive       142
billing_issue            127
fraud_suspicion          118
accès refusé             115

=== COMPLAINTS — status ===
status
open           356
resolved       340
in_progress    194
escalated      163
closed         158

=== MEMBERSHIPS — reason ===
reason
None              706
NaN                70
voluntary          69
owner_request      67
fraud              62
payment_failed     56
inactive           53

=== PAYMENTS — status ===
status
succeeded    3554
failed       1984
pending       581
refunded      375
disputed      216
success       216
suceeded      210
canceled      141

=== USERS — status ===
status
 0.0     1325
 4.0      198
 1.0      179
 3.0      110
 2.0      107
-1.0       38
 NaN       25
 99.0      19


le code suivant permet d'avoir chaque table nettoyée au format .csv
les modifications faites dans les nouvelles tables :
- Les colonnes originales intactes (traçabilité)
- Les colonnes `_clean` normalisées (utilisées pour le scoring)
- Les colonnes de flags (`is_retry`, `self_complaint`, `prefix_flag`)


In [12]:
print(" NETTOYAGE EXHAUSTIF ET DÉFINITIF\n")

# ══════════════════════════════════════════════════════════
# COMPLAINTS — type : traduire accès refusé restant
# ══════════════════════════════════════════════════════════
mapping_type_complet = {
    "accès refusé"        : "access_denied",
    "acces refusé"        : "access_denied",
    "acces refuse"        : "access_denied",
    "paiement échoué"     : "payment_failed",
    "paiement echoue"     : "payment_failed",
    "remboursement"       : "refund_request",
    "fraude"              : "fraud_suspicion",
    "abonnement inactif"  : "subscription_inactive",
    "identifiants erronés": "wrong_credentials",
    "autre"               : "other",
}

df_complaints['type_clean'] = df_complaints['type'].str.lower().str.strip().replace(mapping_type_complet)
print("✅ COMPLAINTS type — valeurs restantes :")
print(df_complaints['type_clean'].value_counts())

# ══════════════════════════════════════════════════════════
# PAYMENTS — status : normaliser les variantes
# ══════════════════════════════════════════════════════════
mapping_payment_status = {
    "success"  : "succeeded",  # variante anglaise
    "suceeded" : "succeeded",  # faute de frappe
    "Succeeded": "succeeded",  # majuscule
    "canceled" : "cancelled",  # variante orthographe
}

df_payments['status'] = df_payments['status'].str.strip().replace(mapping_payment_status)
print("\n PAYMENTS status — valeurs restantes :")
print(df_payments['status'].value_counts())

# ══════════════════════════════════════════════════════════
# MEMBERSHIPS — reason : remplacer NaN textuels
# ══════════════════════════════════════════════════════════
import numpy as np
df_memberships['reason'] = df_memberships['reason'].replace('NaN', np.nan)
print("\n MEMBERSHIPS reason — valeurs restantes :")
print(df_memberships['reason'].value_counts(dropna=False))

# ══════════════════════════════════════════════════════════
# USERS — status : reverse-engineering des codes numériques
# ══════════════════════════════════════════════════════════
# Hypothèse déduite du comportement des données :
# 0   = actif (1325 users, majorité → état normal)
# 1   = inactif (179 users)
# 2   = suspendu (107 users)
# 3   = en attente / pending (110 users)
# 4   = vérifié / premium (198 users)
# -1  = banni (38 users, valeur négative = état exceptionnel)
# 99  = supprimé / anonymisé (19 users, valeur haute = edge case)
# NaN = inconnu

mapping_user_status = {
     0.0: "active",
     1.0: "inactive",
     2.0: "suspended",
     3.0: "pending",
     4.0: "verified",
    -1.0: "banned",
    99.0: "deleted",
}

df_users['status_clean'] = df_users['status'].map(mapping_user_status).fillna("unknown")
print("\n USERS status — reverse-engineering :")
print(df_users[['status', 'status_clean']].value_counts().to_string())

# ══════════════════════════════════════════════════════════
# EXPORT FINAL
# ══════════════════════════════════════════════════════════
from google.colab import files

#df_users.to_csv('users_clean.csv', index=False)
#df_subscriptions.to_csv('subscriptions_clean.csv', index=False)
#df_memberships.to_csv('memberships_clean.csv', index=False)
#df_payments.to_csv('payments_clean.csv', index=False)
#df_complaints.to_csv('complaints_clean.csv', index=False)

print("EXPORT TERMINÉ — téléchargement...")
files.download('users_clean.csv')
files.download('subscriptions_clean.csv')
files.download('memberships_clean.csv')
files.download('payments_clean.csv')
files.download('complaints_clean.csv')

 NETTOYAGE EXHAUSTIF ET DÉFINITIF

✅ COMPLAINTS type — valeurs restantes :
type_clean
access_denied            383
other                    153
wrong_credentials        145
subscription_inactive    143
owner_unresponsive       142
billing_issue            127
fraud_suspicion          118
Name: count, dtype: int64

 PAYMENTS status — valeurs restantes :
status
succeeded    3980
failed       1984
pending       581
refunded      375
disputed      216
cancelled     141
Name: count, dtype: int64

 MEMBERSHIPS reason — valeurs restantes :
reason
None              706
NaN                70
voluntary          69
owner_request      67
fraud              62
payment_failed     56
inactive           53
Name: count, dtype: int64

 USERS status — reverse-engineering :
status  status_clean
 0.0    active          1325
 4.0    verified         198
 1.0    inactive         179
 3.0    pending          110
 2.0    suspended        107
-1.0    banned            38
 99.0   deleted           19
EXPORT TERM

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>